<a href="https://colab.research.google.com/github/Dineeesh2906/Deep-learning---24BAD021/blob/main/24BAD021_cloud_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

# Get the uploaded filename
filename = list(uploaded.keys())[0]

print("Uploaded file:", filename)

# Read CSV
df = pd.read_csv(filename)

print("\nDataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

Saving sentiment-analysis.csv to sentiment-analysis.csv
Uploaded file: sentiment-analysis.csv

Dataset shape: (98, 1)

Column names:
['Text, Sentiment, Source, Date/Time, User ID, Location, Confidence Score']


In [ ]:
# Remove unnecessary spaces from column names
df.columns = df.columns.astype(str).str.strip()

print("Cleaned column names:")
for i, column in enumerate(df.columns):
    print(i, repr(column))

Cleaned column names:
0 'Text, Sentiment, Source, Date/Time, User ID, Location, Confidence Score'


In [ ]:
all_text = " ".join(df.iloc[:, 0].dropna().astype(str))

print(all_text[:500])

"I love this product!", Positive, Twitter, 2023-06-15 09:23:14, @user123, New York, 0.85 "The service was terrible.", Negative, Yelp Reviews, 2023-06-15 11:45:32, user456, Los Angeles, 0.65 "This movie is amazing!", Positive, IMDb, 2023-06-15 14:10:22, moviefan789, London, 0.92 "I'm so disappointed with their customer support.", Negative, Online Forum, 2023-06-15 17:35:11, forumuser1, Toronto, 0.78 "Just had the best meal of my life!", Positive, TripAdvisor, 2023-06-16 08:50:59, foodie22, Paris,


In [ ]:
# Find the column containing the feedback text
text_column = None

for column in df.columns:
    if column.strip().lower() == "text":
        text_column = column
        break

if text_column is None:
    print("ERROR: Text column was not found.")
    print("Available columns:", df.columns.tolist())
else:
    print("Text column found:", repr(text_column))

ERROR: Text column was not found.
Available columns: ['Text, Sentiment, Source, Date/Time, User ID, Location, Confidence Score']


In [ ]:
def mapper_v1(text):
    """
    Basic MapReduce Mapper.
    Emits:
        word -> 1
    """

    words = text.split()

    mapped_data = []

    for word in words:
        mapped_data.append((word, 1))

    return mapped_data

In [ ]:
def shuffle_sort(mapped_data):
    """
    Simulates Hadoop's Shuffle and Sort phase.
    Groups identical keys together.
    """

    grouped_data = {}

    for word, count in mapped_data:

        if word not in grouped_data:
            grouped_data[word] = []

        grouped_data[word].append(count)

    # Sort alphabetically by word
    grouped_data = dict(sorted(grouped_data.items()))

    return grouped_data

In [ ]:
def reducer_v1(grouped_data):
    """
    Reducer adds all counts belonging to the same word.
    """

    result = {}

    for word, counts in grouped_data.items():

        result[word] = sum(counts)

    return result

In [ ]:
def mapreduce_v1(text):

    # ----------------
    # MAP
    # ----------------
    mapped_data = mapper_v1(text)

    # ----------------
    # SHUFFLE + SORT
    # ----------------
    grouped_data = shuffle_sort(mapped_data)

    # ----------------
    # REDUCE
    # ----------------
    result = reducer_v1(grouped_data)

    return result

In [ ]:
result_v1 = mapreduce_v1(all_text)

print("MapReduce Version 1 completed successfully.")
print("Number of unique words:", len(result_v1))

MapReduce Version 1 completed successfully.
Number of unique words: 547


In [ ]:
sorted_v1 = sorted(
    result_v1.items(),
    key=lambda x: x[1],
    reverse=True
)

print("TOP 20 WORDS - FIRST VERSION")
print("=" * 40)

for word, count in sorted_v1[:20]:
    print(f"{word:20} {count}")

TOP 20 WORDS - FIRST VERSION
Positive,            53
Negative,            43
"The                 42
this                 29
is                   26
was                  26
Review,              21
Online               19
the                  18
"I                   17
a                    17
and                  17
at                   17
"This                16
to                   16
I                    15
customer             15
of                   15
with                 15
Store,               13


In [ ]:
sample = df.iloc[0, 0]

print("INPUT:")
print(sample)

INPUT:
"I love this product!", Positive, Twitter, 2023-06-15 09:23:14, @user123, New York, 0.85


In [ ]:
print("\nMAPPER OUTPUT:")
print("=" * 40)

mapper_output = mapper_v1(sample)

for word, count in mapper_output:
    print(f"{word}\t{count}")


MAPPER OUTPUT:
"I	1
love	1
this	1
product!",	1
Positive,	1
Twitter,	1
2023-06-15	1
09:23:14,	1
@user123,	1
New	1
York,	1
0.85	1


In [ ]:
import re

stop_words = {
    "a", "an", "and", "are", "as", "at",
    "be", "by", "for", "from",
    "has", "have", "he", "her",
    "in", "is", "it", "its",
    "of", "on", "or",
    "that", "the", "this",
    "to", "was", "were",
    "will", "with",
    "i", "im", "ive",
    "you", "your",
    "they", "their",
    "we", "our",
    "my"
}

def mapper_v2(text):

    # Convert to lowercase
    text = text.lower()

    # Remove punctuation
    words = re.findall(r"[a-z]+", text)

    mapped = []

    for word in words:

        # Remove stop words
        if word not in stop_words:
            mapped.append((word, 1))

    return mapped

In [ ]:
def reducer_v2(grouped):

    result = {}

    for word, counts in grouped.items():
        result[word] = sum(counts)

    return result

In [ ]:
def mapreduce_v2(text):

    # Mapper
    mapped = mapper_v2(text)

    # Shuffle and Sort
    grouped = shuffle_sort(mapped)

    # Reducer
    result = reducer_v2(grouped)

    return result

In [ ]:
result_v2 = mapreduce_v2(all_text)

print("Improved MapReduce execution completed.")
print("Number of unique words:", len(result_v2))

Improved MapReduce execution completed.
Number of unique words: 318


In [ ]:
sorted_v2 = sorted(
    result_v2.items(),
    key=lambda x: x[1],
    reverse=True
)

print("TOP 20 WORDS - IMPROVED VERSION")
print("=" * 40)

for word, count in sorted_v2[:20]:
    print(f"{word:20} {count}")

TOP 20 WORDS - IMPROVED VERSION
positive             53
negative             43
user                 24
website              23
review               21
online               19
customer             16
store                16
had                  13
product              13
service              13
m                    12
sydney               12
food                 11
london               11
terrible             11
berlin               10
disappointed         10
experience           10
new                  10


In [ ]:
print("FIRST VERSION")
print("=" * 40)

for word, count in sorted_v1[:10]:
    print(f"{word:20} {count}")

print("\nIMPROVED VERSION")
print("=" * 40)

for word, count in sorted_v2[:10]:
    print(f"{word:20} {count}")

FIRST VERSION
Positive,            53
Negative,            43
"The                 42
this                 29
is                   26
was                  26
Review,              21
Online               19
the                  18
"I                   17

IMPROVED VERSION
positive             53
negative             43
user                 24
website              23
review               21
online               19
customer             16
store                16
had                  13
product              13


In [ ]:
original = "I love this product"
modified = "I hate this product"

old_result = mapreduce_v2(original)
new_result = mapreduce_v2(modified)

print("Original:", old_result)
print("Modified:", new_result)

Original: {'love': 1, 'product': 1}
Modified: {'hate': 1, 'product': 1}


In [ ]:
words = set(old_result) | set(new_result)

print("\nCHANGES")
print("=" * 30)

for word in sorted(words):

    old = old_result.get(word, 0)
    new = new_result.get(word, 0)

    if old != new:
        print(f"{word}: {old} -> {new} ({new-old:+d})")


CHANGES
hate: 0 -> 1 (+1)
love: 1 -> 0 (-1)
